# CIDA-Rescue — рой, который спасает людей вместе

Здание-лабиринт. Тяжёлых пострадавших несут только вместе, завалы разбирают только вместе, пожар обходят,
двери внезапно заваливает. Каждый дрон видит лишь окно вокруг себя.

**Конвейер:**
1. базовый рой с общей картой учится с нуля (летать, искать, носить);
2. от его лучшей версии три одинаковых дообучения: **без связи** (просто дольше), **обычное внимание**, **CIDA** —
   одинаковый бюджет, разница только в связи, поэтому сравнение честное;
3. таблица по уровням сложности и HTML-демо: одна сцена — три роя рядом.

Runtime → **A100**. Всё сохраняется на Google Drive; после отключения просто запустите ноутбук заново —
готовые этапы пропустятся.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys, json
if not os.path.exists("/content/CIDA-plugin"):
    !git clone https://github.com/Kairatzh/CIDA-plugin.git /content/CIDA-plugin
%cd /content/CIDA-plugin
!git pull --ff-only
!git log --oneline -1
from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/cida_rescue_runs"
os.makedirs(RUNS, exist_ok=True)

import torch
gpu = torch.cuda.get_device_properties(0)
mem_gb = gpu.total_memory / 2**30
# число миров под память карты: A100 40/80 ГБ — 16384, L4 24 ГБ — 8192, T4 15 ГБ — 4096
N_ENVS = 16384 if mem_gb > 30 else 8192 if mem_gb > 20 else 4096
print(f"GPU: {gpu.name}, {mem_gb:.0f} ГБ → {N_ENVS} миров")
if "A100" not in gpu.name and "H100" not in gpu.name:
    print("ВНИМАНИЕ: это не A100 — обучение будет в разы медленнее. Runtime → Change runtime type → A100.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
WORLD = f"--world rescue --n-drones 6 --min-drones 3 --n-victims 3 --max-steps 300 --share-coverage --n-envs {N_ENVS}"

def run(name, flags):
    if os.path.exists(f"{RUNS}/{name}/result.json"):
        print("готово, пропускаю:", name); return
    cmd = f"{sys.executable} -u -m cida_swarm.train {WORLD} {flags} --name {name} --out {RUNS}/{name}"
    print(">>>", cmd, flush=True)
    get_ipython().system(cmd)
    if not os.path.exists(f"{RUNS}/{name}/result.json"):
        raise RuntimeError(f"{name} не завершился — смотрите лог выше")

## 1. Базовый рой (~1 час, точнее — по скорости в логе)

Учится с нуля на смеси миров всех уровней сложности. Лучшая версия сохраняется в `model_best.pt`.

**Стоп-сигнал, чтобы не жечь кредиты:** случайный рой выносит около 1% людей (`rescued_frac 0.01`).
Если к 150-й итерации (`[eval] it 150`) `rescued_frac` в строке `[eval]` всё ещё ниже 0.03 — остановите ячейку (■)
и пришлите лог: значит, рой не ухватил переноску, и это надо чинить, а не ждать.
Время всего этапа = 3e9 / (agent-steps/s из лога) секунд.

In [ ]:
run("base", "--total-steps 3e9 --eval-every 25 --seed 0")

## 2. Три дообучения с одинаковым бюджетом (~25 минут каждое)

Все стартуют с лучшей версии базового роя. Меньше случайности (`ent-coef`) и маленький шаг обучения —
чтобы не испортить то, что рой уже умеет.

In [ ]:
FT = f"--init-from {RUNS}/base/model_best.pt --lr 1e-4 --ent-coef 0.001 --critic-warmup 3 --total-steps 1.5e9 --eval-every 25 --seed 0"
run("ft_none", f"{FT} --comm none")
run("ft_attn", f"{FT} --comm attn")
run("ft_cida", f"{FT} --comm cida")

## 2b. Связь v2 (~25 минут каждое)

Первая версия связи не дала выигрыша: радиус был 0.3 поля — дрон слышал только тех, кого почти видел сам.
Теперь: радиосеть на всё здание, к услышанному добавлено **направление на говорящего**
(«держу тяжёлого — лети сюда»), а новые модули связи учатся в 10 раз быстрее остального роя.
Тот же бюджет и тот же старт, что у `ft_none`, поэтому сравнение с ним честное.

In [ ]:
run("ft2_attn", f"{FT} --comm attn")
run("ft2_cida", f"{FT} --comm cida")

## 3. CIDA-рой v3 — продукт (~2 часа; точнее = 5e9 / agent-steps/s из лога)

Каждый дрон — отдельный агент: **своя память** (GRU), **своя постоянная личность** (обучаемая, у каждого номера своя),
своя роль в миссии, своё восприятие. Друг с другом дроны связаны только радио CIDA; услышанное попадает в память.

Мир ничего не даёт даром:
* **нет общей карты** — каждый помнит только своё;
* **нет готовой дорожки к выходу** — путь назад надо помнить или узнать у других;
* **вес человека знает только тот, кто его держит** — звать помощь нужно по радио;
* **опасный человек охотится на дронов** — нужны тактики: обойти, отвлечь, разделиться;
* **не ходить толпой** — свободный дрон, прижавшийся к другому свободному дрону вне входа, получает небольшой штраф;
* **автокуррикулум** — чаще выпадают уровни, где рой справляется примерно в половине случаев.

**Глаза и ноги готовы:** базовый рой (уже умеет летать, искать и носить) заморожен. Учится только общий слой CIDA — память, личности, роли и радио — плюс выход действий, через который этот слой управляет телом. Слой подключается с нуля, поэтому рой начинает ровно с умений базового и учится только координации.

**Подсказки убираются постепенно** (как строительные леса): в начале почти все миры с общей картой, дорожкой к выходу и видимым весом — рой работает как базовый; за первые 60% обучения доля таких миров плавно падает до нуля, и память с радио учатся их заменять. Строка `[eval]` всегда меряет мир **без** подсказок — поэтому в начале она низкая и растёт по мере обучения.

**Стоп-сигнал:** если к 100-й итерации `rescued_frac` в обычных строках `it ...` ниже 0.30 — остановите и пришлите лог.

In [ ]:
WORLD = WORLD.replace(" --share-coverage", "")   # общей карты нет — только своя память и радио
HARD = ("--memory --personas --roles --crowd-penalty 0.001 --no-route-hint --hidden-weight "
        "--hazard-chase 0.5 --curriculum --hint-anneal 0.6 --minibatch-size 131072 --epochs 2")
run("v3_cida", f"--init-from {RUNS}/base/model_best.pt --lr 1e-4 --ent-coef 0.003 --critic-warmup 3 --freeze-base "
               f"--total-steps 5e9 --eval-every 25 --seed 0 {HARD} --comm cida")

Почему не вынесены те, кого не вынесли, ходит ли рой толпой и насколько он опирается на радио:

In [ ]:
!cd /content/CIDA-plugin && {sys.executable} -m cida_swarm.diagnose {RUNS}/v3_cida/model_best.pt --labels "CIDA v3"

## 4. Арена: спасатели против охотников (самоигра, ~3 часа)

Вместо запрограммированного злодея по зданию ходит **обучаемая команда охотников** — 1–3 агента со своей памятью,
личностями и своим радио (радио спасателей они не слышат). Охотник выводит из строя спасателя вплотную;
**двое спасателей рядом оглушают охотника** на 12 шагов — это ответная тактика. Во вход охотники не заходят.

* Обе команды учатся **одновременно, играя друг против друга**: награда охотников противоположна спасательной.
* **Лига** (как у AlphaStar): раз в 20 итераций снимок обеих команд в лигу; в 20% итераций одна команда играет
  против старой версии соперника — чтобы не забывать, как побеждать прежние тактики.
* **Генератор трудных карт**: здания, где спасатели провалились, копятся в банке и возвращаются с мутациями
  (новый завал в двери, человек в другой комнате).
* Спасатели стартуют с CIDA-роя v3 («глаза и ноги» заморожены, слой CIDA учится), охотники — с того же мозга.

В `[eval]`: `rescued_frac` — против текущих охотников (может и падать: соперник умнеет),
`vs_first_hunters` — против самых первых охотников (должен расти: спасатели становятся сильнее).

In [ ]:
ARENA = ("--memory --personas --roles --crowd-penalty 0.001 --no-route-hint --hidden-weight --curriculum "
         "--hunters 3 --hard-bank 32768 --minibatch-size 131072 --epochs 2")
run("arena", f"--init-from {RUNS}/v3_cida/model_best.pt --freeze-base --lr 1e-4 --ent-coef 0.003 --critic-warmup 5 "
             f"--total-steps 5e9 --eval-every 25 --seed 0 {ARENA} --comm cida")

Демо арены — те же здания, охотники против CIDA-роя:

In [ ]:
out = f"{RUNS}/arena_demo.html"
!cd /content/CIDA-plugin && {sys.executable} -m cida_swarm.demo {RUNS}/arena/model_best.pt --labels "CIDA против охотников" --scenes 6 --difficulty 2 3 4 --out {out}
from IPython.display import HTML, display
display(HTML(open(out, encoding="utf-8").read()))

## 5. Итоги

In [ ]:
names = {"base": "базовый (общая карта)", "ft_none": "+ дообучение без связи",
         "ft_attn": "+ обычное внимание (v1)", "ft_cida": "+ CIDA (v1)",
         "ft2_attn": "+ обычное внимание (v2)", "ft2_cida": "+ CIDA (v2)",
         "v3_none": "v3: память, без связи", "v3_attn": "v3: память + внимание", "v3_cida": "v3: память + CIDA", "arena": "арена: против охотников"}
print("| Рой | Вынесено людей | Все вынесены | Шагов | Дронов потеряно | Ур.1 | Ур.2 | Ур.3 | Ур.4 |")
print("|---|---|---|---|---|---|---|---|---|")
for key, label in names.items():
    path = f"{RUNS}/{key}/result.json"
    if not os.path.exists(path): continue
    b = json.load(open(path))["best"]
    lv = [f"{b['by_level'].get(str(l), {}).get('rescued_frac', float('nan')):.1%}" for l in range(1, 5)]
    print(f"| {label} | {b['rescued_frac']:.1%} | {b['all_rescued']:.1%} | {b['length']:.0f} | "
          f"{b['drones_lost']:.1%} | " + " | ".join(lv) + " |")

## 6. Демо

Одна сцена — CIDA-рой v3. Файл `rescue_demo.html` сохраняется на Drive: скачайте и откройте в браузере
(работает без интернета, подходит для проектора).

In [ ]:
out = f"{RUNS}/rescue_demo.html"
!{sys.executable} -m cida_swarm.demo {RUNS}/v3_cida/model_best.pt     --labels "CIDA" --scenes 6 --difficulty 3 4 4 --out {out}
from IPython.display import HTML, display
display(HTML(open(out, encoding="utf-8").read()))